In [1]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(mlflow.get_tracking_uri())

C:\Users\user\Documents\bc-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


http://127.0.0.1:5000


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

mlflow.set_experiment("h1-basic-logging")

2026/10/08 21:25:17 INFO mlflow.tracking.fluent: Experiment with name 'h1-basic-logging' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/user/Documents/bc-mlops/mlartifacts/1', creation_time=1791469517857, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791469517857, lifecycle_stage='active', name='h1-basic-logging', tags={}, trace_location=None, workspace='default'>

In [3]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

params = {"n_estimators": 200, "max_depth": 8, "random_state": 42}

In [6]:
with mlflow.start_run(run_name="rf-baseline"):
    mlflow.set_tags({"stage": "baseline", "author": "Devyna Sonya Palupi Sanjaya"})
    mlflow.log_params(params)

    model = RandomForestClassifier(**params).fit(X_tr, y_tr)
    proba = model.predict_proba(X_val)[:, 1]

    mlflow.log_metric("val_f1", f1_score(y_val, proba > 0.5))
    mlflow.log_metric("val_auc", roc_auc_score(y_val, proba))
    mlflow.sklearn.log_model(model, name="model", input_example=X_val[:5],
                             skops_trusted_types=["sklearn.tree._tree.Tree"])

🏃 View run rf-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/3d0a2a7de99c4d1eb99ffe7ee6669220
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [8]:
from sklearn.ensemble import GradientBoostingClassifier

def log_run(run_name, model, stage):
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({"stage": stage, "author": "Devyna Sonya Palupi Sanjaya", "model_family": type(model).__name__})
        mlflow.log_params(model.get_params())
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        mlflow.log_metric("val_f1", f1_score(y_val, proba > 0.5))
        mlflow.log_metric("val_auc", roc_auc_score(y_val, proba))
        mlflow.sklearn.log_model(model, name="model", input_example=X_val[:5],
                                 skops_trusted_types=["sklearn.tree._tree.Tree"])

In [9]:
log_run("rf-deep", RandomForestClassifier(n_estimators=700, max_depth=3, random_state=42), "tuning")
log_run("gb-baseline", GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42), "model-selection")

🏃 View run rf-deep at: http://127.0.0.1:5000/#/experiments/1/runs/685c05cae04943eeaedfd3dd0e28c56a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run gb-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/907d72b7cd464ac8b7b164e1bfe413b9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [10]:
log_run("rf-deep", RandomForestClassifier(n_estimators=900, max_depth=3, random_state=42), "tuning")
log_run("gb-baseline", GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42), "model-selection")

🏃 View run rf-deep at: http://127.0.0.1:5000/#/experiments/1/runs/5b074bf2b77049a5b6b95bf37b88d2ea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run gb-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/960467c967324050b2611accbf531479
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [11]:
log_run("rf-deep", RandomForestClassifier(n_estimators=800, max_depth=3, random_state=42), "tuning")
log_run("gb-baseline", GradientBoostingClassifier(n_estimators=500, learning_rate=0.1, random_state=42), "model-selection")

🏃 View run rf-deep at: http://127.0.0.1:5000/#/experiments/1/runs/1f4cc050dff94453841d9149eb7b0ca2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run gb-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/706f696bda134a9cb3f01f80c951ddc7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [12]:
from sklearn.svm import SVC
mlflow.set_experiment("h1-autolog")
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="svc-autolog"):
    svc = SVC(C=1.0, kernel="rbf", probability=True, random_state=42).fit(X_tr, y_tr)
    print("val accuracy:", svc.score(X_val, y_val))

mlflow.sklearn.autolog(disable=True)   # matikan lagi agar run berikutnya tidak ikut ter-autolog

2026/10/08 21:52:38 INFO mlflow.tracking.fluent: Experiment with name 'h1-autolog' does not exist. Creating a new experiment.
C:\Users\user\Documents\bc-mlops\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
2026/10/08 21:52:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
C:\Users\user\Documents\bc-mlops\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in

val accuracy: 0.9298245614035088
🏃 View run svc-autolog at: http://127.0.0.1:5000/#/experiments/2/runs/cfa02d5ac0d446eca53f6caa4e19f835
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [13]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
mlflow.set_experiment("h1-optuna")

def train_and_eval(p):
    m = RandomForestClassifier(**p, random_state=42).fit(X_tr, y_tr)
    return f1_score(y_val, m.predict(X_val))

def objective(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 50, 300),
         "max_depth": trial.suggest_int("max_depth", 2, 16),
         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10)}
    with mlflow.start_run(run_name=f"trial-{trial.number}", nested=True):
        mlflow.log_params(p)
        f1 = train_and_eval(p)
        mlflow.log_metric("val_f1", f1)
    return f1

with mlflow.start_run(run_name="optuna-sweep-01"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_f1", study.best_value)

2026/10/08 21:57:13 INFO mlflow.tracking.fluent: Experiment with name 'h1-optuna' does not exist. Creating a new experiment.


🏃 View run trial-0 at: http://127.0.0.1:5000/#/experiments/3/runs/2d81332c9a3146f0a68f4f5b24052c67
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run trial-1 at: http://127.0.0.1:5000/#/experiments/3/runs/30de71868f6b4ccb8ef6c7a20d161f4b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run trial-2 at: http://127.0.0.1:5000/#/experiments/3/runs/46e016033370443983d0a39539352d02
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run trial-3 at: http://127.0.0.1:5000/#/experiments/3/runs/4f8ec5d1ab8f480f890b8a68e9844281
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run trial-4 at: http://127.0.0.1:5000/#/experiments/3/runs/d9d4e327830c444aa1dac3b4a7d71d26
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run trial-5 at: http://127.0.0.1:5000/#/experiments/3/runs/755e7ece404e412a810c0ed830be6810
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run trial-6 at: http://127.0.0.1:5000/#

In [14]:
runs = mlflow.search_runs(
    experiment_names=["h1-basic-logging"],
    filter_string="metrics.val_f1 > 0.9",
    order_by=["metrics.val_f1 DESC"],
    max_results=10,
)
print(runs[["run_id", "tags.mlflow.runName", "metrics.val_f1", "metrics.val_auc"]])
best_run_id = runs.iloc[0]["run_id"]
print("best:", best_run_id)

                             run_id tags.mlflow.runName  metrics.val_f1  \
0  706f696bda134a9cb3f01f80c951ddc7         gb-baseline        0.965986   
1  960467c967324050b2611accbf531479         gb-baseline        0.965986   
2  907d72b7cd464ac8b7b164e1bfe413b9         gb-baseline        0.965986   
3  3d0a2a7de99c4d1eb99ffe7ee6669220         rf-baseline        0.965517   
4  d2c7d09630aa43ba8c18e30e4c6a1a2c         rf-baseline        0.965517   
5  1f51cbbdc7ec4f86a7c567d89449b431         rf-baseline        0.965517   
6  1f4cc050dff94453841d9149eb7b0ca2             rf-deep        0.958333   
7  5b074bf2b77049a5b6b95bf37b88d2ea             rf-deep        0.958333   
8  685c05cae04943eeaedfd3dd0e28c56a             rf-deep        0.958333   

   metrics.val_auc  
0         0.993717  
1         0.990741  
2         0.989749  
3         0.994048  
4         0.994048  
5         0.994048  
6         0.991402  
7         0.991402  
8         0.991402  
best: 706f696bda134a9cb3f01f80c951ddc7


In [15]:
from mlflow import MlflowClient
client = MlflowClient()
name = "h1-breast-cancer-clf"

mv = mlflow.register_model(f"runs:/{best_run_id}/model", name)
client.update_model_version(name, mv.version, description="Run terbaik h1-basic-logging")
client.set_model_version_tag(name, mv.version, "validation_status", "passed")
client.set_registered_model_alias(name, "challenger", mv.version)
client.set_registered_model_alias(name, "champion", mv.version)

model = mlflow.pyfunc.load_model(f"models:/{name}@champion")
print(model.predict(X_val[:5]))

Successfully registered model 'h1-breast-cancer-clf'.
2026/10/08 22:04:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 706f696bda134a9cb3f01f80c951ddc7 has no artifacts at artifact path 'model', registering model based on models:/m-bf32e6b6ba804ed489c7f31cfe7c33a3 instead
2026/10/08 22:04:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: h1-breast-cancer-clf, version 1
Created version '1' of model 'h1-breast-cancer-clf'.


[0 1 0 0 0]


In [ ]:
A